**PREPROCESAMIENTO DE DATOS EEG — VERSIÓN OPTIMIZADA**

Cambios respecto a la versión original: sin instalar paquetes en cada corrida (celda de verificación de versiones); los canales problemáticos se **interpolan** antes de ICA y del epocado; épocas guardadas como `-epo.fif`; corrección de línea base aplicada explícitamente y verificada; diagnóstico de épocas descartadas (duración, onsets, intervalos, `drop_log`).

**Protocolo:** Sección 4.3.2 - Flujo de trabajo estandarizado

**Versión:**
  - PROCESAMIENTO POR LOTES CON VERIFICACIÓN DE CALIDAD
  - MÚLTIPLES FORMATOS DE ALMACENAMIENTO
  - CORRECCIÓN DE PARPADEOS

**Genera:**
  - Procesamiento automático de TODOS los archivos .cnt en una carpeta
  - EEG limpios
  - Épocas completas (9s) en .dat y .fif
  - Épocas segmentadas (Imaginería y Percepción) en .dat(ASCII), .edf(Estándar) y .fif (Para procesamiento en Phyton)

**LIBRERÍAS UTILIZADAS:**
- mne: Librería especializada para análisis de EEG/MEG
  * mne.io.read_raw_cnt(): Carga archivos Neuroscan .cnt
  * mne.Epochs(): Crea épocas alrededor de eventos
  * raw.filter(): Filtrado digital de señales
  * raw.set_eeg_reference(): Re-referenciación de canales
  * raw.export(): Exporta a formato EDF
  
- numpy: Operaciones numéricas eficientes
  * np.std(), np.ptp(): Estadísticos para detección de ruido
  * np.corrcoef(): Correlación entre canales vecinos
  
- scipy.signal: Procesamiento de señales
  * find_peaks(): Detección de parpadeos en EOG
  
- pickle: Serialización para guardar datos en formato DAT
- json: Almacenamiento de metadatos y bitácoras
- glob: Búsqueda de archivos por patrón

**CODIFICACIÓN DE EVENTOS:**
- Colores: 1→11, 2→12, 3→13, 4→14
- Formas:  1→21, 2→22, 3→23, 4→24

**FASE 1: Control de calidad inicial**

✅ Inspección visual multicanal (guarda imágenes)

✅ Identificación de canales problemáticos (amplitud, correlación, espectro)

✅ Documentación en bitácora estandarizada (JSON)

✅ Criterio de calidad: 80% señal utilizable

**FASE 2: Limpieza básica**

✅ Re-referenciación a mastoides (A1, A2)

✅ Filtrado: pasa-alto 0.5 Hz, pasa-bajo 100 Hz, notch 60 Hz; remuestreo a 250 Hz

✅ Detección de canales malos (criterios combinados)

✅ Interpolación de canales malos (solo EEG de cuero cabelludo) ANTES de ICA y del epocado

✅ Artefactos de parpadeo: ICA (FastICA, 20 componentes; `find_bads_eog` con EOGL/EOGR, umbral 3.0) detecta y excluye los componentes de parpadeo; los canales EOG no entran a las épocas



**FASE 3: Segmentación y epocado**

✅ Percepción: -100ms a +5000ms

✅ Imaginería: +5000ms a +9000ms

✅ Corrección de línea base: -100ms a 0ms (aplicada explícitamente y verificada; las épocas de percepción e imaginería la heredan de la época maestra)

✅ Guardado en FIF (`-epo.fif`), DAT y EDF (EDF de épocas = promedio evocado)

# **Estructura del Directorio**


    Procesado/
    ├── 01_Bitacoras/  <-(JSONs, logs y gráficas de control de calidad)
    ├── 02_EEG_Continuo_Limpio/ <-(Señal continua filtrada sin cortes, ideal para quienes quieran re-epocar)
    │   ├── EDF/
    │   └── FIF/
    ├── 03_Epocas_Completas_9s/ <-(Época maestra de todo el ensayo)
    │   ├── DAT/
    │   ├── EDF/
    │   └── FIF/
    └── 04_Epocas_Segmentadas/  <-(Los recortes listos para Machine Learning)
    ├── Imagineria/
    │   ├── DAT/
    │   ├── EDF/
    │   └── FIF/
    └── Percepcion/
        ├── DAT/
        ├── EDF/
        └── FIF/


## Instalación de dependencias

En Colab cada sesión nueva arranca con un entorno limpio, por eso esta celda instala primero las librerías con `!pip install` (versiones fijas del último re-corrido) y después verifica e imprime las versiones instaladas. Si las librerías ya están, pip lo detecta rápido y no las vuelve a bajar.

In [13]:
# ==============================================
# SETUP: INSTALACIÓN DE LIBRERÍAS (Colab)
# ==============================================
# En Colab cada sesión nueva arranca con un entorno limpio, por eso este
# notebook instala sus librerías al inicio. Si ya están instaladas, pip
# lo detecta rápido y no las vuelve a bajar. El -q deja la salida corta.
!pip install -q mne==1.13.2 numpy==2.4.6 scipy==1.16.3 matplotlib==3.10.0 edfio==0.4.18

import importlib
import sys

# Versiones de referencia del último re-corrido a 250 Hz (Python 3.13)
VERSIONES_REFERENCIA = {
    'mne': '1.13.2',
    'numpy': '2.4.6',
    'scipy': '1.16.3',
    'matplotlib': '3.10.0',
    'edfio': '0.4.18',
}

print(f"Python: {sys.version.split()[0]} (referencia: 3.13)")
for _nombre, _ref in VERSIONES_REFERENCIA.items():
    try:
        _mod = importlib.import_module(_nombre)
        _ver = getattr(_mod, '__version__', 'instalado')
        print(f"  ✓ {_nombre}: {_ver} (referencia: {_ref})")
    except ImportError:
        print(f"  ✗ {_nombre}: NO INSTALADO (referencia: {_ref})")
print("\n✓ Verificación de versiones completa.")

Python: 3.13.18 (referencia: 3.13)
  ✓ mne: 1.13.2 (referencia: 1.13.2)
  ✓ numpy: 2.4.6 (referencia: 2.4.6)
  ✓ scipy: 1.16.3 (referencia: 1.16.3)
  ✓ matplotlib: 3.10.0 (referencia: 3.10.0)
  ✓ edfio: 0.4.18 (referencia: 0.4.18)

✓ Verificación de versiones completa.


## Inicialización de librerías y montaje de Drive

Importa las librerías que usa el notebook (MNE, NumPy, SciPy, etc.) y monta el Google Drive en `/content/drive` para leer los `.cnt` y escribir los procesados. Recibe: nada. Produce: el Drive montado y las librerías listas para usar.

In [14]:
# ==============================================
# INCIALIZACIÓN DE LIBRERÍAS
# ==============================================
import os
import numpy as np
import mne
import json
import pickle
import glob
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
from scipy import signal, stats
import re

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configuración del protocolo (clase `ConfigTesis`)

Centraliza todo lo que el protocolo necesita: rutas de entrada/salida, codificación de eventos (colores 1–4 → 11–14, formas 1–4 → 21–24), criterios de calidad, filtros (0.5/100 Hz, notch 60 Hz), remuestreo a 250 Hz, parámetros de ICA y de epocado. Recibe: nada. Produce: el objeto global `config` que usan todas las fases. Su método `archivo_ya_procesado` dice si un archivo ya tiene su época maestra guardada, para no reprocesarlo.

In [15]:
# ==========================================================
# CONFIGURACIÓN - BASADO EN PROTOCOLO TESIS (Sección 4.3.2)
# ==========================================================

class ConfigTesis:
    def __init__(self):
        # RUTAS PRINCIPALES
        self.carpeta_base = "/content/drive/Othercomputers/Mi portátil/Doctorado/VISUAL_DECODING/Neuroscan_Data"
        self.carpeta_raw = os.path.join(self.carpeta_base, "DATA")  # <-- Datos crudos (eeg)
        self.carpeta_procesado = os.path.join(self.carpeta_base, "Procesado")
        self.carpeta_logs = os.path.join(self.carpeta_procesado, "01_Bitacoras")

        # ESTRUCTURA JERÁRQUICA PARA PUBLICACIÓN
        self.rutas = {
            'eeg_continuo': os.path.join(self.carpeta_procesado, "02_EEG_Continuo_Limpio"),
            'completo_9s': os.path.join(self.carpeta_procesado, "03_Epocas_Completas_9s"),
            'percepcion': os.path.join(self.carpeta_procesado, "04_Epocas_Segmentadas", "Percepcion"),
            'imagineria': os.path.join(self.carpeta_procesado, "04_Epocas_Segmentadas", "Imagineria")
        }

        # Crear toda la estructura con subcarpetas por formato (DAT, EDF, FIF)
        os.makedirs(self.carpeta_logs, exist_ok=True)
        for tipo, ruta_principal in self.rutas.items():
            formatos = ['FIF', 'EDF'] if tipo == 'eeg_continuo' else ['FIF', 'DAT', 'EDF']
            for formato in formatos:
                os.makedirs(os.path.join(ruta_principal, formato), exist_ok=True)

        # ==============================
        # CODIFICACIÓN DE EVENTOS
        # ==============================
        self.mapeo_eventos = {
            'colores': {
                1: 11,  # Rojo
                2: 12,  # Amarillo
                3: 13,  # Verde
                4: 14   # Azul
            },
            'formas': {
                1: 21,  # Cuadrado
                2: 22,  # Triángulo
                3: 23,  # Círculo
                4: 24   # Estrella
            }
        }

        # Eventos finales por tipo
        self.eventos_finales = {
            'colores': [11, 12, 13, 14],
            'formas': [21, 22, 23, 24]
        }

        # ======================================
        # PARÁMETROS DE CALIDAD (FASE 1)
        # ======================================
        self.criterio_calidad = {
            'senal_utilizable_min': 0.80,  # 80% señal utilizable
            'canales_problematicos_max': 3,  # Máximo 3 canales problemáticos
            'std_max': 100,  # µV - umbral para identificación de ruido
            'amp_max': 500,  # µV - amplitud pico-pico máxima
        }

        # ======================================
        # PARÁMETROS DE LIMPIEZA (FASE 2)
        # ======================================
        self.referencia = ['A1', 'A2']

        self.filtros = {
            'pasa_alto': 0.5,
            'pasa_bajo': 100,     # Para preservar banda Gamma
            'notch': [60]         # Notch estricto en 60 Hz
        }

        # Frecuencia de análisis (remuestreo tras el filtrado anti-aliasing)
        self.sfreq_objetivo = 250  # Hz

        # Configuraciones para ICA
        self.ica_config = {
            'n_components': 20,
            'method': 'fastica',
            'random_state': 97
        }

        self.criterios_canales_malos = {
            'amplitud_min': -150,
            'amplitud_max': 150,
            'correlacion_min': 0.4,
            'std_max': 80,
        }

        self.canales_eog = ['EOGL', 'EOGR']

        # Montaje 10-20 extendido (el nombre anterior esta deprecado en MNE 1.14).
        # Se asigna al raw tras la carga para poder interpolar canales malos.
        self.montaje = 'colin27_1005'

        # Ventana de línea base: -100 a 0 ms (tesis Fase 3: resta del promedio -100 a 0 ms)
        self.baseline = (-0.1, 0.0)

        # Canales que NUNCA se interpolan: EOG, Trigger y mastoides (A1, A2).
        # Solo se interpolan canales EEG de cuero cabelludo.
        self.canales_no_interpolables = ['EOGL', 'EOGR', 'Trigger', 'A1', 'A2']

        # ======================================
        # PARÁMETROS DE EPOCADO (FASE 3)
        # ======================================
        self.epocas = {
            'completo_9s': {
                'tmin': -0.1,
                'tmax': 9.0,
                'baseline': (-0.1, 0.0)
            },
            'percepcion': {
                'tmin': -0.1,
                'tmax': 5.0
            },
            'imagineria': {
                'tmin': 5.0,
                'tmax': 9.0
            }
        }

        # ======================================
        # DETECCIÓN DE TIPO DE TAREA
        # ======================================
        self.patron_tipo = r'^[0-9]+_[A-Z]+_([CF])_[A-Z0-9]+_Data$'

    def archivo_ya_procesado(self, nombre_base):
        """Verifica si la época maestra en FIF ya existe en la carpeta de destino.

        Acepta el nombre viejo (_completo_9s.fif) y el nuevo (_completo_9s-epo.fif),
        para no reprocesar archivos ya guardados con cualquiera de los dos."""
        for sufijo in ("_completo_9s.fif", "_completo_9s-epo.fif"):
            ruta_fif = os.path.join(self.rutas['completo_9s'], 'FIF', f"{nombre_base}{sufijo}")
            if os.path.exists(ruta_fif):
                return True
        return False

# Instanciamos la clase para actualizar la variable global
config = ConfigTesis()

## Funciones de utilidad: eventos y guardado

- `reasignar_eventos_desde_anotaciones(raw, tipo_tarea)`: lee los eventos desde las anotaciones del `.cnt` y reasigna los códigos 1–4 a 11–14 (colores) o 21–24 (formas). Recibe el `raw` y el tipo de tarea; devuelve el `raw` con anotaciones nuevas, la matriz de eventos y el diccionario de eventos.
- `guardar_en_todos_formatos(epochs, nombre_base, tipo_epoca)`: guarda unas épocas en FIF, DAT y EDF (el EDF de épocas es el promedio evocado, porque el EDF es un formato continuo). Recibe las épocas y el nombre base; devuelve las rutas y tamaños de los archivos guardados.

In [16]:
# =======================
# FUNCIONES DE UTILIDAD
# =======================
# ==============================================
# REASIGNACIÓN DE EVENTOS DESDE ANOTACIONES
# ==============================================

def reasignar_eventos_desde_anotaciones(raw, tipo_tarea):
    """
    Lee los eventos desde las ANOTACIONES y los reasigna según la codificación:
    - Colores: 1→11, 2→12, 3→13, 4→14
    - Formas:  1→21, 2→22, 3→23, 4→24
    - Mantiene otros eventos (10, 20, etc.) sin cambios

    ARGUMENTOS:
        raw: objeto Raw de MNE con datos EEG
        tipo_tarea: 'colores' o 'formas'

    RETORNA:
        raw: objeto Raw con anotaciones modificadas
        events: matriz de eventos (n_eventos x 3)
        event_dict: diccionario con mapeo descripción→ID

    NOTA: mne.events_from_annotations() convierte anotaciones a eventos
    """
    print(f"\n  🔄 Reasignando eventos desde anotaciones...")
    print(f"    Tipo: {tipo_tarea.upper()}")

    # raw.annotations: objeto MNE que contiene las anotaciones del archivo
    annotations = raw.annotations
    print(f"    Anotaciones originales: {len(annotations)}")
    print(f"    Primeras 5: {annotations.description[:5]}")

    nuevos_descriptions = []
    reasignados = 0
    otros_eventos = 0

    # Obtener el mapeo según el tipo de tarea
    mapeo = config.mapeo_eventos[tipo_tarea]

    for desc in annotations.description:
        try:
            # int(float(desc)): convierte strings como '1' a entero 1
            val_original = int(float(desc))
        except:
            # Si no es numérico, mantener igual (ej: 'bad segment')
            nuevos_descriptions.append(desc)
            continue

        # REASIGNACIÓN SEGÚN TIPO
        if val_original in mapeo:
            nuevo_val = mapeo[val_original]
            nuevos_descriptions.append(str(nuevo_val))
            reasignados += 1
            print(f"      {val_original} → {nuevo_val}")
        else:
            # Mantener otros eventos (10, 20, 211, etc.)
            nuevos_descriptions.append(desc)
            otros_eventos += 1

    print(f"    Eventos reasignados: {reasignados}")
    print(f"    Otros eventos mantenidos: {otros_eventos}")

    # Crear nuevas anotaciones con MNE.Annotations()
    # onset: tiempo de inicio en segundos
    # duration: duración del evento
    # description: etiqueta del evento
    nuevas_annotations = mne.Annotations(
        onset=annotations.onset,
        duration=annotations.duration,
        description=nuevos_descriptions,
        orig_time=annotations.orig_time
    )

    # raw.set_annotations(): reemplaza las anotaciones en el objeto Raw
    raw.set_annotations(nuevas_annotations)

    # mne.events_from_annotations(): convierte anotaciones a matriz de eventos
    # Crear un mapeo explícito para que MNE no invente IDs secuenciales (1, 2, 3...)
    mapeo_explicito = {desc: int(desc) for desc in np.unique(nuevos_descriptions) if desc.isdigit()}

    # Usar el mapeo explícito
    events, event_dict = mne.events_from_annotations(raw, event_id=mapeo_explicito, verbose=False)

    print(f"    Eventos finales: {sorted(event_dict.keys())}")

    return raw, events, event_dict


# ==============================================
# FUNCIONES DE GUARDADO EN MÚLTIPLES FORMATOS
# ==============================================
"""
    Guarda épocas en todos los formatos soportados:
    - FIF (MNE nativo) - más rápido para cargar en MNE
    - DAT (numpy array via pickle) - portable, fácil de usar en cualquier entorno
    - EDF (formato estándar europeo) - interoperable con otros software

    ARGUMENTOS:
        epochs: objeto Epochs de MNE
        nombre_base: nombre base del archivo
        tipo_epoca: 'completo_9s', 'percepcion', 'imagineria'
        subcarpeta_adicional: subcarpeta específica (ej: 'completo')

    RETORNA:
        diccionario con rutas y tamaños de archivos guardados
"""
def guardar_en_todos_formatos(epochs, nombre_base, tipo_epoca, subcarpeta_adicional=None):
 #   Guarda las épocas en la carpeta correspondiente según su tipo y formato
    if epochs is None or len(epochs) == 0:
        return None

    resultados = {}
    data = epochs.get_data()

    # Identificar la carpeta base usando nuestra nueva configuración
    base_dir = config.rutas[tipo_epoca]

    # Si hay subcarpeta_adicional, se puede usar (aunque actualmente no se usa en la lógica)
    if subcarpeta_adicional:
        # Puedes incorporar lógica aquí si es necesario
        pass

    # 1. Guardar FIF
    # Las épocas se guardan con sufijo -epo.fif (convención MNE; evita el
    # RuntimeWarning de nomenclatura al cargarlas). DAT y EDF mantienen su nombre.
    fif_path = os.path.join(base_dir, 'FIF', f"{nombre_base}_{tipo_epoca}-epo.fif")
    epochs.save(fif_path, overwrite=True)
    resultados['fif'] = {'path': fif_path, 'tamaño_mb': os.path.getsize(fif_path) / (1024 * 1024)}

    # 2. Guardar DAT (Pickle)
    dat_path = os.path.join(base_dir, 'DAT', f"{nombre_base}_{tipo_epoca}.dat")
    datos_completos = {
        'data': data, 'events': epochs.events, 'ch_names': epochs.ch_names,
        'sfreq': epochs.info['sfreq'], 'tmin': epochs.tmin, 'tmax': epochs.tmax,
        'event_id': epochs.event_id, 'tipo_epoca': tipo_epoca
    }
    with open(dat_path, 'wb') as f:
        pickle.dump(datos_completos, f)
    resultados['dat'] = {'path': dat_path, 'tamaño_mb': os.path.getsize(dat_path) / (1024 * 1024)}

    # 3. Guardar EDF (Promedio Evocado) — EDF válido
    # Nota: evoked.save() escribe FIF aunque el archivo se llame .edf; por eso
    # el promedio se convierte a RawArray y se exporta con raw.export(fmt='edf').
    try:
        edf_path = os.path.join(base_dir, 'EDF', f"{nombre_base}_{tipo_epoca}.edf")
        evoked = epochs.average()
        info_edf = mne.create_info(evoked.ch_names, evoked.info['sfreq'], ch_types='eeg')
        raw_evoked = mne.io.RawArray(evoked.data, info_edf, verbose=False)
        raw_evoked.export(edf_path, fmt='edf', overwrite=True)
        resultados['edf'] = {'path': edf_path, 'tamaño_mb': os.path.getsize(edf_path) / (1024 * 1024)}
    except Exception as e:
        print(f"    ⚠️ Error guardando EDF: {str(e)}")

    return resultados

## Fase 1 — Control de calidad inicial (clase `ControlCalidad`)

Revisa cada archivo antes de limpiarlo: inspección visual multicanal (guarda imágenes), identificación de canales problemáticos (amplitud, correlación, espectro) y evaluación de calidad (criterio: 80 % de señal utilizable). Recibe el `raw` y el nombre base; produce el diagnóstico, la bitácora JSON y la decisión de si el archivo procede a limpieza. Su método `registrar_interpolacion` anota qué canales se interpolaron.

In [17]:
# ==============================================
# FASE 1: REVISIÓN Y CONTROL DE CALIDAD INICIAL
# ==============================================

class ControlCalidad:
    #Implementa la Fase 1 del protocolo

    def __init__(self):
        self.bitacora = {
            'fecha': str(datetime.now()),
            'archivos_revisados': [],
            'resumen_general': {}
        }

    def inspeccion_visual_multicanal(self, raw, nombre_archivo):
        """
        Inspección visual multicanal de señal cruda
        Genera gráficas para revisión visual
        raw[:, :]: obtiene todos los datos y tiempos del objeto Raw
        plt.subplots(): crea múltiples subplots para cada canal
        """

        print(f"\n  📊 Inspección visual multicanal...")

        import matplotlib.pyplot as plt

        # raw[:, :] devuelve (datos, tiempos)
        data, times = raw[:, :]
        n_canales = min(20, len(raw.ch_names))  # Limitar a 20 canales para visualización

        # Crear subplots, uno por canal
        fig, axes = plt.subplots(n_canales, 1, figsize=(15, 2*n_canales), sharex=True)

        for i in range(n_canales):
            ax = axes[i] if n_canales > 1 else axes
            # Graficar primeros 10000 puntos (10s a 1000Hz)
            ax.plot(times[:10000], data[i, :10000])
            ax.set_ylabel(raw.ch_names[i], fontsize=8)
            ax.set_yticks([])

        axes[-1].set_xlabel('Tiempo (s)')
        plt.suptitle(f'Inspección inicial - {nombre_archivo}')
        plt.tight_layout()

        # plt.savefig(): guarda la figura como imagen PNG
        img_path = os.path.join(config.carpeta_logs, f"{nombre_archivo}_inspeccion_inicial.png")
        plt.savefig(img_path, dpi=100, bbox_inches='tight')
        plt.close()

        return img_path

    def identificar_canales_problematicos(self, raw):
        """
        Identificación de canales problemáticos y artefactos evidentes
        Usa numpy para cálculos estadísticos:
        - np.min(), np.max(): valores mínimo y máximo
        - np.ptp(): peak-to-peak (rango)
        - np.std(): desviación estándar
        - np.corrcoef(): correlación entre canales
        """
        data, times = raw[:, :]
        canales_problematicos = []
        diagnostico = {}

        for i, ch_name in enumerate(raw.ch_names):
            if ch_name in config.canales_eog:
                continue  # Los EOG se manejan aparte

            señal = data[i, :]

            # Estadísticos básicos
            amp_min = np.min(señal)
            amp_max = np.max(señal)
            amp_pp = np.ptp(señal)  # peak-to-peak amplitude
            std = np.std(señal)      # desviación estándar

            # Correlación con canales vecinos
            # np.corrcoef(): matriz de correlación de Pearson
            correlacion = 1.0
            if i > 0 and i < len(raw.ch_names)-1:
                señal_prev = data[i-1, :]
                señal_next = data[i+1, :]
                corr_prev = np.corrcoef(señal, señal_prev)[0,1]
                corr_next = np.corrcoef(señal, señal_next)[0,1]
                # Usar la máxima correlación absoluta con vecinos
                correlacion = max(abs(corr_prev), abs(corr_next)) if not np.isnan(corr_prev) else 1.0

            diagnostico[ch_name] = {
                'amp_min': float(amp_min),
                'amp_max': float(amp_max),
                'amp_pp': float(amp_pp),
                'std': float(std),
                'correlacion': float(correlacion) if not np.isnan(correlacion) else 1.0
            }

            # Evaluar si es problemático según umbrales
            es_problematico = False
            razones = []

            if amp_min < config.criterios_canales_malos['amplitud_min']:
                es_problematico = True
                razones.append(f"amplitud mínima baja: {amp_min:.1f} µV")

            if amp_max > config.criterios_canales_malos['amplitud_max']:
                es_problematico = True
                razones.append(f"amplitud máxima alta: {amp_max:.1f} µV")

            if std > config.criterios_canales_malos['std_max']:
                es_problematico = True
                razones.append(f"ruido excesivo (std={std:.1f} µV)")

            if not np.isnan(correlacion) and correlacion < config.criterios_canales_malos['correlacion_min']:
                es_problematico = True
                razones.append(f"baja correlación con vecinos: {correlacion:.2f}")

            if es_problematico:
                canales_problematicos.append({
                    'canal': ch_name,
                    'razones': razones,
                    'metricas': diagnostico[ch_name]
                })

        return canales_problematicos, diagnostico

    def evaluar_calidad_inicial(self, raw, canales_problematicos):
        """
        Evalúa si los datos cumplen el criterio de calidad (80% utilizable)
        según protocolo de tesis
        """
        total_canales = len([ch for ch in raw.ch_names if ch not in config.canales_eog])
        canales_utiles = total_canales - len(canales_problematicos)
        porcentaje_util = canales_utiles / total_canales if total_canales > 0 else 0

        calidad_aceptable = porcentaje_util >= config.criterio_calidad['senal_utilizable_min']

        return {
            'total_canales': total_canales,
            'canales_problematicos': len(canales_problematicos),
            'canales_utiles': canales_utiles,
            'porcentaje_util': float(porcentaje_util),
            'calidad_aceptable': calidad_aceptable,
            'procede_limpieza': calidad_aceptable
        }

    def registrar_interpolacion(self, nombre_archivo, canales_interpolados):
        """Añade a la entrada de bitácora del archivo qué canales se interpolaron."""
        for entrada in self.bitacora['archivos_revisados']:
            if entrada['archivo'] == nombre_archivo:
                entrada['resultados_inspeccion']['canales_interpolados'] = list(canales_interpolados)
                break
        bitacora_path = os.path.join(config.carpeta_logs, "bitacora_calidad.json")
        with open(bitacora_path, 'w') as f:
            json.dump(self.bitacora, f, indent=2, default=str)
        return bitacora_path

    def documentar_en_bitacora(self, nombre_archivo, resultados):
        """
        Documenta observaciones en bitácora estandarizada (formato JSON)
        json.dump(): serializa diccionario a archivo JSON
        """
        entrada_bitacora = {
            'archivo': nombre_archivo,
            'timestamp': str(datetime.now()),
            'resultados_inspeccion': resultados
        }

        self.bitacora['archivos_revisados'].append(entrada_bitacora)

        bitacora_path = os.path.join(config.carpeta_logs, "bitacora_calidad.json")
        with open(bitacora_path, 'w') as f:
            json.dump(self.bitacora, f, indent=2, default=str)

        return bitacora_path

## Fase 2 — Limpieza básica de señal (clase `LimpiezaSenal`)

Limpia la señal en este orden: re-referencia a mastoides (A1, A2), filtrado pasa-alto 0.5 Hz / pasa-bajo 100 Hz / notch 60 Hz, remuestreo a 250 Hz, interpolación de los canales malos (solo EEG de cuero cabelludo) y ICA (FastICA, 20 componentes, `find_bads_eog` con EOGL/EOGR, umbral 3.0) con validación visual. Recibe el `raw`; devuelve el `raw` limpio y los ajustes de ICA.

In [18]:
# ==============================================
# FASE 2: LIMPIEZA BÁSICA DE SEÑAL
# ==============================================
from mne.preprocessing import ICA
import matplotlib.pyplot as plt
import os

class LimpiezaSenal:
    """Implementa la Fase 2 del protocolo con ICA y validaciones visuales"""

    def __init__(self):
        self.ajustes_aplicados = []

    def aplicar_referencia(self, raw):
        print(f"  🔄 Aplicando referencia a mastoides...")
        if all(ch in raw.ch_names for ch in config.referencia):
            raw.set_eeg_reference(ref_channels=config.referencia)
        else:
            raw.set_eeg_reference('average')
        return raw

    def aplicar_filtros(self, raw):
        print(f"  🔧 Aplicando filtros digitales...")
        # Notch estricto en 60 Hz
        raw.notch_filter(config.filtros['notch'], picks=['eeg', 'eog'], verbose=False)
        # Filtro pasa-banda ancho para preservar Gamma
        raw.filter(config.filtros['pasa_alto'], config.filtros['pasa_bajo'],
                   picks=['eeg', 'eog'], fir_design='firwin', verbose=False)
        return raw

    def aplicar_resample(self, raw):
        print(f"  🔄 Remuestreando a {config.sfreq_objetivo} Hz...")
        # El pasa-bajo a 100 Hz ya aplicado actúa como filtro anti-aliasing
        # (Nyquist a 250 Hz = 125 Hz > 100 Hz, la banda Gamma 30-100 Hz se preserva)
        if raw.info['sfreq'] > config.sfreq_objetivo:
            raw.resample(config.sfreq_objetivo, verbose=False)
            print(f"    ✓ Nueva frecuencia de muestreo: {raw.info['sfreq']} Hz")
        else:
            print(f"    ✓ Ya está a {raw.info['sfreq']} Hz, sin cambios.")
        return raw

    def interpolar_canales_malos(self, raw, canales_problematicos):
        """Marca los canales problemáticos (solo EEG de cuero cabelludo) en
        raw.info['bads'] y los interpola con raw.interpolate_bads().

        Se ejecuta ANTES de la referencia, los filtros y el ICA.
        Los canales EOG, Trigger y las mastoides (A1, A2) nunca se interpolan.
        Requiere que el montaje (colin27_1005) ya esté asignado al raw.

        Seguridad (corrección 2026-11-19): la interpolación por splines
        falla con "array must not contain infs or NaNs" si algún canal EEG
        no tiene posición en el montaje (p. ej., A1/A2 pueden no estar en
        colin27_1005). Por eso:
          1) se completan las posiciones faltantes desde standard_1005;
          2) solo se interpolan canales con posición válida;
          3) la interpolación se prueba en una copia y solo se aplica si
             el resultado no tiene NaN/inf; si falla, el archivo continúa
             SIN interpolar (como el código original) y queda registrado.

        Devuelve (raw_a_usar, lista_de_canales_interpolados)."""
        import numpy as np
        import mne
        a_interpolar = [c['canal'] for c in canales_problematicos
                        if c['canal'] not in config.canales_no_interpolables
                        and c['canal'] in raw.ch_names]
        if not a_interpolar:
            print(f"    ✓ Sin canales EEG que interpolar.")
            return raw, []
        # 1) Completar posiciones faltantes de canales EEG desde standard_1005
        #    (colin27_1005 puede no definir A1/A2; sin posición, la spline
        #    produce NaN y rompe el archivo).
        try:
            montaje_std = mne.channels.make_standard_montage('standard_1005')
            pos_std = montaje_std.get_positions()['ch_pos']
            sin_pos = []
            for idx, ch in enumerate(raw.ch_names):
                loc = raw.info['chs'][idx]['loc']
                if not np.all(np.isfinite(loc[:3])):
                    if ch in pos_std:
                        raw.info['chs'][idx]['loc'][:3] = pos_std[ch]
                    else:
                        sin_pos.append(ch)
            if sin_pos:
                print(f"    ⚠️ Canales sin posición en ningún montaje: {sin_pos}")
        except Exception as e:
            print(f"    ⚠️ No se pudieron completar posiciones ({e}); se sigue con las del montaje actual.")
        # 2) Solo canales con posición válida.
        con_posicion = []
        sin_posicion = []
        for ch in a_interpolar:
            idx = raw.ch_names.index(ch)
            pos = raw.info['chs'][idx]['loc'][:3]
            if np.all(np.isfinite(pos)):
                con_posicion.append(ch)
            else:
                sin_posicion.append(ch)
        if sin_posicion:
            print(f"    ⚠️ Sin posición en el montaje (no se interpolan): {sin_posicion}")
        if not con_posicion:
            print("    ⚠️ Ningún canal malo tiene posición; se continúa sin interpolar.")
            return raw, []
        print(f"    🔧 Interpolando {len(con_posicion)} canal(es): {con_posicion}")
        # 3) Probar en una copia; solo aplicar si el resultado es válido.
        raw_prueba = raw.copy()
        raw_prueba.info['bads'] = list(con_posicion)
        try:
            raw_prueba.interpolate_bads(reset_bads=True, verbose=False)
            datos = raw_prueba.get_data()
            if not np.all(np.isfinite(datos)):
                raise ValueError("la interpolación produjo NaN/inf en los datos")
        except Exception as e:
            print(f"    ⚠️ No se pudo interpolar ({e}); se continúa sin interpolar este archivo.")
            return raw, []
        print(f"    ✓ Canales interpolados y restaurados.")
        return raw_prueba, con_posicion

    def aplicar_ica_y_validar(self, raw, nombre_base):
        """
        Aplica ICA de forma automatizada, elimina componentes de parpadeo
        y genera gráficas de validación en la carpeta de logs.
        """
        print(f"  🧠 Aplicando ICA para limpieza de artefactos...")
        ajustes = {'componentes_excluidos': []}
        inds_eog = []  # se define aquí para que plot_overlay no falle si no hay EOG

        # 1. Ajuste del ICA
        ica = ICA(n_components=config.ica_config['n_components'],
                  method=config.ica_config['method'],
                  random_state=config.ica_config['random_state'],
                  max_iter='auto')

        # Filtramos temporalmente para que el ICA aprenda mejor (no afecta la señal original)
        raw_for_ica = raw.copy().filter(l_freq=1.0, h_freq=None, verbose=False)
        ica.fit(raw_for_ica, verbose=False)

        # 2. Búsqueda automatizada de parpadeos usando canales EOG
        if all(ch in raw.ch_names for ch in ['EOGL', 'EOGR']):
            inds_eog, scores_eog = ica.find_bads_eog(raw, ch_name=['EOGL', 'EOGR'], threshold=3.0, verbose=False)
            ica.exclude = inds_eog
            ajustes['componentes_excluidos'] = inds_eog
            print(f"    ✓ Componentes oculares identificados: {inds_eog}")
        else:
            print("    ⚠️ Canales EOG no encontrados, saltando limpieza ocular automatizada.")

        # 3. GENERACIÓN DE VALIDACIONES VISUALES (Bitácoras)
        print(f"  📊 Generando gráficas de validación...")

        # A. Gráfica de Overlay (Antes y Después del ICA)
        fig_overlay = ica.plot_overlay(raw.copy(), exclude=inds_eog, show=False)
        overlay_path = os.path.join(config.carpeta_logs, f"{nombre_base}_01_ICA_Overlay.png")
        fig_overlay.savefig(overlay_path, bbox_inches='tight', dpi=150)
        plt.close(fig_overlay)

        # 4. Aplicar la limpieza real a los datos crudos
        raw_clean = raw.copy()
        ica.apply(raw_clean)

        # B. Gráfica PSD (Verificación de preservación Gamma)
        fig_psd = raw_clean.compute_psd(fmax=100).plot(show=False)
        psd_path = os.path.join(config.carpeta_logs, f"{nombre_base}_02_PSD_Gamma.png")
        fig_psd.savefig(psd_path, bbox_inches='tight', dpi=150)
        plt.close(fig_psd)

        # C. Gráfica de Señal Limpia (Multicanal) para revisión experta
        data_clean, times_clean = raw_clean[:, :]
        n_canales = min(20, len(raw_clean.ch_names))

        fig_clean, axes_clean = plt.subplots(n_canales, 1, figsize=(15, 2*n_canales), sharex=True)
        for i in range(n_canales):
            ax = axes_clean[i] if n_canales > 1 else axes_clean
            ax.plot(times_clean[:10000], data_clean[i, :10000], color='black', linewidth=0.8)
            ax.set_ylabel(raw_clean.ch_names[i], fontsize=8)
            ax.set_yticks([])

        axes_clean[-1].set_xlabel('Tiempo (s)')
        plt.suptitle(f'Inspección Post-Limpieza (ICA) - {nombre_base}')
        plt.tight_layout()

        clean_path = os.path.join(config.carpeta_logs, f"{nombre_base}_03_EEG_Limpio.png")
        fig_clean.savefig(clean_path, dpi=100, bbox_inches='tight')
        plt.close(fig_clean)

        self.ajustes_aplicados.append(ajustes)
        return raw_clean, ajustes

## Fase 3 — Segmentación y epocado (clase `SegmentacionEpocado`)

Crea la época maestra de 9 s (tmin −0.1 s, tmax 9.0 s) con corrección de línea base (−0.1 a 0.0 s) aplicada explícitamente y verificada, y de ella recorta percepción (−0.1–5.0 s) e imaginería (5.0–9.0 s). Recibe el `raw` limpio, los eventos y el tipo de tarea; produce las tres épocas y un diagnóstico de retención (duración, onsets, intervalos entre eventos y `drop_log`). Su método `guardar_todas_las_epocas` guarda cada tipo en FIF, DAT y EDF.

In [19]:
# ==============================================
# FASE 3: SEGMENTACIÓN Y EPOCADO
# ==============================================

class SegmentacionEpocado:
    """Implementa la Fase 3 del protocolo: Epocado y Recorte para ML"""

    def __init__(self):
        self.epocas_creadas = {}

    def extraer_epocas(self, raw, events, event_dict, tipo_tarea, nombre_base):
        print(f"\n  📊 Extrayendo épocas y centrando línea base...")
        event_ids = {str(e): e for e in config.eventos_finales[tipo_tarea] if str(e) in event_dict.keys()}

        if len(event_ids) == 0:
            print(f"    ⚠️ No hay eventos válidos para epocar")
            return None

        canales_eeg = [ch for ch in raw.ch_names if ch not in config.canales_eog + ['Trigger']]
        print(f"      Canales en las épocas ({len(canales_eeg)}): {canales_eeg}")

        # ---- Diagnóstico de épocas (retención) ----
        # Duración del registro, onsets e intervalos entre eventos de estímulo.
        # Deja visible cuántas épocas se descartan y por qué (drop_log).
        sfreq = raw.info['sfreq']
        duracion_raw = float(raw.times[-1]) if len(raw.times) else 0.0
        ids_estimulo = sorted(event_ids.values())
        eventos_estimulo = events[np.isin(events[:, 2], ids_estimulo)]
        n_eventos_estimulo = len(eventos_estimulo)
        onsets_s = eventos_estimulo[:, 0] / sfreq if n_eventos_estimulo else np.array([])
        longitud_epoca = config.epocas['completo_9s']['tmax'] - config.epocas['completo_9s']['tmin']
        if n_eventos_estimulo > 1:
            intervalos = np.diff(onsets_s)
            print(f"      Duración del registro: {duracion_raw:.1f} s | sfreq: {sfreq} Hz")
            print(f"      Eventos de estímulo: {n_eventos_estimulo} | onsets: {onsets_s[0]:.2f}–{onsets_s[-1]:.2f} s")
            print(f"      Intervalo entre eventos: mediana {np.median(intervalos):.2f} s, "
                  f"mín {np.min(intervalos):.2f} s, máx {np.max(intervalos):.2f} s")
            print(f"      Longitud de la época maestra: {longitud_epoca:.1f} s "
                  f"(tmin={config.epocas['completo_9s']['tmin']}, tmax={config.epocas['completo_9s']['tmax']})")
        elif n_eventos_estimulo == 1:
            print(f"      Duración del registro: {duracion_raw:.1f} s | solo 1 evento de estímulo")
        else:
            print(f"      Duración del registro: {duracion_raw:.1f} s | sin eventos de estímulo")
        resultados = {}

        epochs_completo = mne.Epochs(
            raw, events, event_id=event_ids,
            tmin=config.epocas['completo_9s']['tmin'],
            tmax=config.epocas['completo_9s']['tmax'],
            baseline=config.epocas['completo_9s']['baseline'],
            preload=True, picks=canales_eeg, verbose=True, reject=None
        )

        # ---- Resumen de épocas descartadas (drop_log) ----
        n_retenidas = len(epochs_completo)
        n_descartadas = n_eventos_estimulo - n_retenidas
        pct_retenidas = (n_retenidas / n_eventos_estimulo * 100) if n_eventos_estimulo else 0.0
        print(f"      Épocas retenidas: {n_retenidas} de {n_eventos_estimulo} eventos de estímulo "
              f"({pct_retenidas:.1f}%) | descartadas: {n_descartadas}")
        if n_descartadas > 0:
            from collections import Counter
            motivos = Counter()
            for entrada in epochs_completo.drop_log:
                for motivo in entrada:
                    motivos[motivo] += 1
            print(f"      Motivos de descarte (drop_log): {dict(motivos)}")
            print(f"      (Una época se descarta sobre todo si su ventana sale del registro; "
                  f"revisa los onsets y la duración arriba.)")

        # ---- Corrección de línea base explícita y verificación ----
        # El constructor ya aplica baseline=(-0.1, 0.0); se aplica de nuevo de forma
        # explícita para que la corrección quede garantizada en los datos guardados.
        # La media del segmento de línea base ya es ~0, así que no hay doble corrección.
        epochs_completo.apply_baseline(baseline=config.baseline)
        print(f"      Baseline aplicada: {epochs_completo.baseline}")
        # Verificación: la media del segmento -0.1 a 0.0 s debe ser ~0
        idx_bl = (epochs_completo.times >= config.baseline[0]) & (epochs_completo.times <= config.baseline[1])
        media_bl = float(np.abs(epochs_completo.get_data()[:, :, idx_bl].mean()))  # |media con signo|; la media de |valores| nunca es ~0
        print(f"      Verificación baseline: |media| del segmento -0.1 a 0.0 s = {media_bl:.2e} V "
              f"({'✓ ~0' if media_bl < 1e-9 else '⚠️ revisa'})")

        if len(epochs_completo) > 0:
            resultados['completo_9s'] = {
                'epochs': epochs_completo, 'n_epochs': len(epochs_completo), 'shape': list(epochs_completo.get_data().shape)
            }
            print(f"      ✓ Época Maestra creada y centrada: {len(epochs_completo)} ensayos")

            # Percepción e imaginería son recortes de la época maestra: heredan la
            # corrección de línea base ya aplicada; NO se les aplica una segunda corrección.
            epochs_percepcion = epochs_completo.copy().crop(tmin=config.epocas['percepcion']['tmin'], tmax=config.epocas['percepcion']['tmax'])
            resultados['percepcion'] = {
                'epochs': epochs_percepcion, 'n_epochs': len(epochs_percepcion), 'shape': list(epochs_percepcion.get_data().shape)
            }
            print(f"      ✓ Percepción (4s) lista para tensores: {list(epochs_percepcion.get_data().shape)}")

            epochs_imagineria = epochs_completo.copy().crop(tmin=config.epocas['imagineria']['tmin'], tmax=config.epocas['imagineria']['tmax'])
            resultados['imagineria'] = {
                'epochs': epochs_imagineria, 'n_epochs': len(epochs_imagineria), 'shape': list(epochs_imagineria.get_data().shape)
            }
            print(f"      ✓ Imaginería (4s) lista para tensores: {list(epochs_imagineria.get_data().shape)}")

        return resultados

    def guardar_todas_las_epocas(self, resultados, nombre_base):
        """Guarda todas las épocas en los tres formatos"""
        archivos_generados = {}
        for tipo_epoca, datos in resultados.items():
            if datos is None:
                continue
            epochs = datos['epochs']
            subcarpeta = 'completo' if tipo_epoca == 'completo_9s' else None

            guardados = guardar_en_todos_formatos(epochs, nombre_base, tipo_epoca, subcarpeta_adicional=subcarpeta)

            archivos_generados[tipo_epoca] = {
                'n_epochs': datos['n_epochs'],
                'shape': datos['shape'],
                'archivos': guardados
            }
        return archivos_generados

## Detección del tipo de tarea

`función detectar_tipo_tarea(nombre_archivo)`: decide si un archivo es de colores o de formas por su nombre (patrón `XX_INICIALES_T_S_B`, con T = C o F), con respaldos si el patrón no coincide exacto. Recibe el nombre del archivo; devuelve `'colores'`, `'formas'` o `None` si no se pudo detectar.

In [20]:
# ==============================================
# DETECCIÓN DE TIPO DE TAREA
# ==============================================

def detectar_tipo_tarea(nombre_archivo):
    """
    Detecta si el archivo es de colores o formas por su nombre
    Usa expresiones regulares (re.match) para buscar patrones

    re.match(): busca patrón al inicio del string
    grupos de captura: ([CF]) captura C o F
    """
    nombre_base = os.path.splitext(nombre_archivo)[0]

    match = re.match(config.patron_tipo, nombre_base)

    if match:
        tipo = match.group(1)  # Extrae el grupo capturado (C o F)
        return 'colores' if tipo.upper() == 'C' else 'formas'

    # Fallbacks si el patrón no coincide exactamente
    if '_C_' in nombre_base:
        return 'colores'
    elif '_F_' in nombre_base:
        return 'formas'

    if 'C_' in nombre_base or nombre_base.endswith('_C'):
        return 'colores'
    elif 'F_' in nombre_base or nombre_base.endswith('_F'):
        return 'formas'

    print(f"⚠️ No se pudo detectar tipo para: {nombre_archivo}")
    return None

## Fase 4 — Guardado del EEG continuo limpio

`función guardar_eeg_limpio(raw, nombre_base)`: guarda la señal continua ya limpia en FIF y en EDF (formato estándar europeo) dentro de `02_EEG_Continuo_Limpio`. Recibe el `raw` limpio y el nombre base; devuelve las rutas y tamaños de los archivos guardados.

In [21]:
# ==============================================
# GUARDAR EEG LIMPIO EN FORMATO EDF
# ==============================================

def guardar_eeg_limpio(raw, nombre_base):
    """Guarda el EEG continuo limpio en FIF y EDF"""
    print(f"\n  💾 Guardando EEG continuo limpio...")
    resultados = {}
    base_dir = config.rutas['eeg_continuo']

    # Guardar en FIF
    fif_path = os.path.join(base_dir, 'FIF', f"{nombre_base}_continuo.fif")
    raw.save(fif_path, overwrite=True)
    resultados['fif'] = {'path': fif_path, 'tamaño_mb': os.path.getsize(fif_path) / (1024 * 1024)}
    print(f"    ✓ FIF guardado ({resultados['fif']['tamaño_mb']:.2f} MB)")

    # Guardar en EDF
    try:
        edf_path = os.path.join(base_dir, 'EDF', f"{nombre_base}_continuo.edf")
        raw.export(edf_path, fmt='edf', overwrite=True)
        resultados['edf'] = {'path': edf_path, 'tamaño_mb': os.path.getsize(edf_path) / (1024 * 1024)}
        print(f"    ✓ EDF guardado ({resultados['edf']['tamaño_mb']:.2f} MB)")
    except Exception as e:
        print(f"    ⚠️ Error exportando a EDF: {str(e)}")

    return resultados

## Flujo principal: procesar todos los archivos

`función procesar_segun_protocolo()`: recorre todos los `.cnt` de la carpeta `DATA` y aplica, en orden, Fase 1 (calidad), reasignación de eventos, Fase 2 (limpieza), guardado del EEG continuo y Fase 3 (epocado y guardado). Salta los archivos ya procesados y los que no cumplen el criterio de calidad. Recibe: nada (usa `config`); produce los EEG limpios, las épocas y las bitácoras en `Procesado/`, y el resumen en `resultados_protocolo.json`.

In [22]:
# ==============================================
# FLUJO PRINCIPAL
# ==============================================

def procesar_segun_protocolo():
    """Ejecuta el flujo completo del protocolo"""

    print("\n" + "="*80)
    print(" 🧠 PROCESAMIENTO EEG - PROTOCOLO TESIS (Sección 4.3.2) ")
    print("="*80)
    print("\n📁 Estructura de carpetas:")
    for nombre, ruta in config.rutas.items():
        print(f"   - {nombre}: {ruta}")

    # Inicializar clases
    calidad = ControlCalidad()
    limpieza = LimpiezaSenal()
    segmentacion = SegmentacionEpocado()

    # Buscar archivos .cnt en la carpeta de datos crudos (DATA)
    archivos_cnt = glob.glob(os.path.join(config.carpeta_raw, "*.cnt"))

    if not archivos_cnt:
        print(f"\n❌ No se encontraron archivos .cnt en: {config.carpeta_raw}")
        return

    print(f"\n📂 Archivos encontrados: {len(archivos_cnt)}")

    resultados_finales = {
        'protocolo': '4.3.2',
        'fecha': str(datetime.now()),
        'codificacion_eventos': '1-4_mapeados_a_11-14_21-24',
        'archivos': []
    }

    ya_existentes = 0

    for archivo_cnt in archivos_cnt:
        nombre_archivo = os.path.basename(archivo_cnt)
        nombre_base = os.path.splitext(nombre_archivo)[0]
        nombre_base = nombre_base.replace(' ', '_').replace('(', '').replace(')', '')

        # Lógica de verificación corregida
        if config.archivo_ya_procesado(nombre_base):
            print(f"\n⏭️  Saltando: {nombre_archivo} (Ya existe en la carpeta de procesados)")
            ya_existentes += 1
            continue

        print(f"\n{'='*60}")
        print(f"Procesando: {nombre_archivo}")
        print('='*60)

        tipo_tarea = detectar_tipo_tarea(nombre_archivo)
        if not tipo_tarea:
            print(f"  ⚠️ Skipping - no se pudo determinar tipo")
            continue

        print(f"  Tipo: {tipo_tarea.upper()}")

        try:
            # ======================================
            # FASE 1: CARGA Y CONTROL DE CALIDAD
            # ======================================
            print("\n🔍 FASE 1: Control de calidad inicial")

            # mne.io.read_raw_cnt(): carga archivo Neuroscan .cnt
            raw = mne.io.read_raw_cnt(archivo_cnt, preload=True, verbose=False)

            # Montaje 10-20 extendido (necesario para interpolar canales malos;
            # on_missing='ignore' deja sin posición a EOGL/EOGR/Trigger, que no están
            # en el montaje estándar y nunca se interpolan).
            raw.set_montage(config.montaje, on_missing='ignore')

            img_path = calidad.inspeccion_visual_multicanal(raw, nombre_base)
            canales_problematicos, diagnostico = calidad.identificar_canales_problematicos(raw)
            evaluacion = calidad.evaluar_calidad_inicial(raw, canales_problematicos)

            print(f"  Calidad: {evaluacion['porcentaje_util']:.1%} utilizable")
            print(f"  Canales problemáticos: {len(canales_problematicos)}")

            calidad.documentar_en_bitacora(nombre_archivo, {
                'evaluacion': evaluacion,
                'canales_problematicos': canales_problematicos,
                'inspeccion_visual': img_path
            })

            if not evaluacion['procede_limpieza']:
                print(f"  ⚠️ Datos no cumplen criterio de calidad (mínimo 80%)")
                resultados_finales['archivos'].append({
                    'archivo': nombre_archivo,
                    'estado': 'rechazado_calidad',
                    'evaluacion': evaluacion
                })
                continue

            # Interpolar canales malos (Fase 1 -> Fase 2): solo EEG de cuero
            # cabelludo; EOG, Trigger y mastoides (A1, A2) nunca se interpolan.
            # Se hace ANTES de la referencia, los filtros, el ICA y el epocado.
            raw, canales_interpolados = limpieza.interpolar_canales_malos(raw, canales_problematicos)
            calidad.registrar_interpolacion(nombre_archivo, canales_interpolados)

            # ======================================
            # FASE 2: LIMPIEZA BÁSICA DE SEÑAL
            # ======================================
            print("\n🧹 FASE 2: Limpieza básica de señal")

            raw = limpieza.aplicar_referencia(raw)
            raw = limpieza.aplicar_filtros(raw)
            raw = limpieza.aplicar_resample(raw)

            # ======================================
            # REASIGNACIÓN DE EVENTOS (CRÍTICO)
            # ======================================
            # Se hace DESPUÉS del remuestreo: los eventos se extraen de las
            # anotaciones (onsets en segundos) y MNE los convierte a índices
            # de muestra con la sfreq actual. Si se extrajeran antes del
            # remuestreo (1000 Hz) y se aplicaran al raw ya remuestreado
            # (250 Hz), los índices quedarían 4x inflados y la mayoría de
            # las épocas se descartarían como TOO_SHORT.
            print("\n🔢 Reasignando eventos desde anotaciones...")
            raw, events, event_dict = reasignar_eventos_desde_anotaciones(raw, tipo_tarea)

            # Aplicamos ICA y generamos las validaciones visuales
            raw, ajustes_ica = limpieza.aplicar_ica_y_validar(raw, nombre_base)

            # Guardar EEG continuo limpio en EDF y FIF
            eeg_edf = guardar_eeg_limpio(raw, nombre_base)

            # ======================================
            # FASE 3: SEGMENTACIÓN Y EPOCADO
            # ======================================
            print("\n📊 FASE 3: Segmentación y epocado")

            epocas_resultados = segmentacion.extraer_epocas(raw, events, event_dict, tipo_tarea, nombre_base)

            archivos_generados = {}
            if epocas_resultados:
                archivos_generados = segmentacion.guardar_todas_las_epocas(epocas_resultados, nombre_base)

                resultado_archivo = {
                    'archivo': nombre_archivo,
                    'tipo': tipo_tarea,
                    'estado': 'completado',
                    'codificacion_eventos': '1-4_mapeados_a_11-14_21-24',
                    'evaluacion_calidad': evaluacion,
                    'canales_interpolados': canales_interpolados,
                    'ajustes_ica': ajustes_ica,
                    'eeg_limpio_edf': eeg_edf,
                    'epocas': {
                        tipo: {
                            'n_epochs': datos['n_epochs'],
                            'shape': datos['shape']
                        } for tipo, datos in epocas_resultados.items()
                    },
                    'archivos_generados': archivos_generados
                }

                resultados_finales['archivos'].append(resultado_archivo)

                print(f"\n✅ {nombre_archivo} procesado exitosamente")
                if 'percepcion' in epocas_resultados:
                    print(f"   Percepción: {epocas_resultados['percepcion']['n_epochs']} épocas")
                if 'imagineria' in epocas_resultados:
                    print(f"   Imaginería: {epocas_resultados['imagineria']['n_epochs']} épocas")
                if 'completo_9s' in epocas_resultados:
                    print(f"   Completo 9s: {epocas_resultados['completo_9s']['n_epochs']} épocas")

        except Exception as e:
            print(f"\n❌ Error procesando {nombre_archivo}: {str(e)}")
            resultados_finales['archivos'].append({
                'archivo': nombre_archivo,
                'estado': 'error',
                'error': str(e)
            })

    # ======================================
    # RESUMEN FINAL
    # ======================================
    print("\n" + "="*80)
    print(" 📊 RESUMEN FINAL - PROTOCOLO COMPLETADO ")
    print("="*80)

    resultados_path = os.path.join(config.carpeta_logs, "resultados_protocolo.json")
    with open(resultados_path, 'w') as f:
        json.dump(resultados_finales, f, indent=2, default=str)

    total = len(resultados_finales['archivos'])
    completados = sum(1 for a in resultados_finales['archivos'] if a['estado'] == 'completado')
    rechazados = sum(1 for a in resultados_finales['archivos'] if a['estado'] == 'rechazado_calidad')
    errores = sum(1 for a in resultados_finales['archivos'] if a['estado'] == 'error')

    print(f"\n📁 Resumen de archivos:")
    print(f"   ✅ Procesados en esta sesión: {completados}")
    print(f"   ⏭️  Previamente procesados: {ya_existentes}")
    print(f"   ⚠️  Rechazados (calidad): {rechazados}")
    print(f"   ❌ Errores: {errores}")

    print(f"\n📂 Datos finales guardados en:")
    print(f"   {config.carpeta_procesado}")
    print(f"\n📋 Bitácoras guardadas en:")
    print(f"   {config.carpeta_logs}")
    print(f"\n📌 Codificación de eventos:")
    print(f"   Colores: 1→11, 2→12, 3→13, 4→14")
    print(f"   Formas:  1→21, 2→22, 3→23, 4→24")

    print("\n" + "="*80)
    print(" ✅ PROTOCOLO DE TESIS COMPLETADO EXITOSAMENTE ")
    print("="*80)

    return resultados_finales

# ==============================================
# EJECUTAR
# ==============================================

if __name__ == "__main__":
    resultados = procesar_segun_protocolo()

Se truncaron las últimas líneas 5000 del resultado de transmisión.
      4 → 14
      2 → 12
      4 → 14
      1 → 11
      2 → 12
      1 → 11
      2 → 12
      4 → 14
      2 → 12
      4 → 14
      1 → 11
      4 → 14
      1 → 11
      4 → 14
      3 → 13
      4 → 14
      3 → 13
      2 → 12
      1 → 11
      2 → 12
      3 → 13
      4 → 14
      2 → 12
      1 → 11
      3 → 13
      1 → 11
      2 → 12
      1 → 11
      3 → 13
      1 → 11
      2 → 12
      3 → 13
      2 → 12
      1 → 11
      1 → 11
      3 → 13
      2 → 12
      1 → 11
      2 → 12
      1 → 11
      3 → 13
      2 → 12
      3 → 13
      1 → 11
      2 → 12
      1 → 11
    Eventos reasignados: 58
    Otros eventos mantenidos: 61
    Eventos finales: ['10', '11', '12', '13', '14', '211']
  🧠 Aplicando ICA para limpieza de artefactos...
Use rng= instead of random_state= in new code
    ✓ Componentes oculares identificados: []
  📊 Generando gráficas de validación...
Applying ICA to Raw instance
    Tr